In [1]:
import numpy as np
import statsmodels.api as sm
from pathlib import Path
import pickle
import yaml

In [2]:
from setup import (
    filepath,
    testing_labels,
    spectral_range,
    testing_settling_time as settling_time,
    rise_time,
    gas_cell_volume,
    calibration_type,
    predictor_type,
    configuration,
)
from processing import (
    is_system_settled,
    scaling_from_pid_to_cell,
    crossing_time_slope,
    format_number,
)

### Load testing datasets

In [3]:
times = []
voltages = []
absorbances = []
flowrates = []
setpoints = []
are_settled = []
are_analyte_free = []

wavenumber = None
spectral_resolution = None
gas_1 = None
gas_2 = None
split_ratio = None
window_type = None
correction_type = None
signal_type = None

if predictor_type != "absorbance":
    raise ValueError(f"Unsupported predictor type: {predictor_type}.")

for label in testing_labels:
    dataset = np.load(filepath / f"{label}.npz")
    metadata = yaml.safe_load((filepath / f"{label}.yaml").read_text())

    if spectral_resolution is None:
        spectral_resolution = metadata["sampling"]["spectral_resolution"]
    elif spectral_resolution != metadata["sampling"]["spectral_resolution"]:
        raise ValueError(f"Spectral resolution mismatch for {label}.")

    time = dataset["time"]
    spectrum = dataset["spectrum"]

    if wavenumber is None:
        wavenumber = dataset["wavenumber"]
        spectral_mask = (wavenumber >= spectral_range[0]) & (
            wavenumber <= spectral_range[1]
        )
    elif wavenumber.shape != dataset["wavenumber"].shape:
        raise ValueError(f"Wavenumber mismatch for {label}.")
    elif not np.allclose(wavenumber, dataset["wavenumber"], atol=1e-3):
        raise ValueError(f"Wavenumber mismatch for {label}.")

    if gas_1 is None:
        gas_1 = str(dataset["gas_1"])
    elif gas_1 != str(dataset["gas_1"]):
        raise ValueError(f"Gas 1 mismatch for {label}.")

    if gas_2 is None:
        gas_2 = str(dataset["gas_2"])
    elif gas_2 != str(dataset["gas_2"]):
        raise ValueError(f"Gas 2 mismatch for {label}.")

    if split_ratio is None:
        split_ratio = float(dataset["split_ratio"])
    elif split_ratio != float(dataset["split_ratio"]):
        raise ValueError(f"Split ratio mismatch for {label}.")

    if dataset["type"] == "twins":
        if window_type is None:
            window_type = str(dataset["window_type"])
        elif window_type != str(dataset["window_type"]):
            raise ValueError(f"Window type mismatch for {label}.")

        if correction_type is None:
            correction_type = str(dataset["correction_type"])
        elif correction_type != str(dataset["correction_type"]):
            raise ValueError(f"Correction type mismatch for {label}.")

        if signal_type is None:
            signal_type = str(dataset["signal_type"])
        elif signal_type != str(dataset["signal_type"]):
            raise ValueError(f"Signal type mismatch for {label}.")

    setpoint_0 = dataset["setpoint_0"]
    setpoint_1 = dataset["setpoint_1"]
    setpoint_2 = dataset["setpoint_2"]
    setpoint = np.stack(
        [
            np.mean(setpoint_0, axis=1),
            np.mean(setpoint_1, axis=1),
            np.mean(setpoint_2, axis=1),
        ],
        axis=1,
    )

    flowrate_0 = dataset["flowrate_0"]
    flowrate_1 = dataset["flowrate_1"]
    flowrate_2 = dataset["flowrate_2"]
    flowrate = np.stack(
        [
            np.mean(flowrate_0, axis=1),
            np.mean(flowrate_1, axis=1),
            np.mean(flowrate_2, axis=1),
        ],
        axis=1,
    )

    voltage_pid_1 = dataset["voltage_pid_1"] / 1e6  # (V)
    voltage_pid_2 = dataset["voltage_pid_2"] / 1e6  # (V)
    voltage = np.stack(
        [np.mean(voltage_pid_1, axis=1), np.mean(voltage_pid_2, axis=1)],
        axis=1,
    )

    time_offset = np.max(np.concatenate(times)) if times else 0
    time = np.min(time, axis=1) + time_offset

    spectrum = spectrum[:, spectral_mask]

    is_settled = is_system_settled(time, setpoint_1, setpoint_2, settling_time)
    is_analyte_free = (
        np.all((setpoint_0 != 0) & (setpoint_1 == 0) & (setpoint_2 == 0), axis=1)
        & is_settled
    )

    if not np.any(is_analyte_free):
        raise ValueError(f"No analyte-free spectra found for {label}.")

    absorbance = -np.log10(spectrum / np.mean(spectrum[is_analyte_free, :], axis=0))

    times.append(time)
    voltages.append(voltage)
    absorbances.append(absorbance)
    flowrates.append(flowrate)
    setpoints.append(setpoint)
    are_settled.append(is_settled)
    are_analyte_free.append(is_analyte_free)

times = np.concatenate(times, axis=0)
voltages = np.concatenate(voltages, axis=0)
absorbances = np.concatenate(absorbances, axis=0)
flowrates = np.concatenate(flowrates, axis=0)
setpoints = np.concatenate(setpoints, axis=0)
are_settled = np.concatenate(are_settled, axis=0)
are_analyte_free = np.concatenate(are_analyte_free, axis=0)

gases = (gas_1, gas_2)
wavenumber = wavenumber[spectral_mask]

In [4]:
number_spectra, number_wavenumbers = np.shape(absorbances)
number_gases = len(gases)
gas_pair = "-".join(gas.lower() for gas in gases)
gas_labels = {"isopropanol": "iPrOH", "ethanol": "EtOH"}

print(f"Datasets: {', '.join(testing_labels)}")
print(f"Gases: {gas_labels[gas_1]}, {gas_labels[gas_2]}")
print(
    f"Spectra: {number_spectra} "
    f"({np.sum(are_settled)} are settled, {np.sum(are_analyte_free)} are analyte free)"
)
print(f"Wavenumbers: {number_wavenumbers} at {spectral_resolution} cm^-1 resolution")

Datasets: GRASP26_72F
Gases: iPrOH, EtOH
Spectra: 882 (402 are settled, 210 are analyte free)
Wavenumbers: 30 at 20 cm^-1 resolution


### Load calibrated concentrations

In [5]:
concentrations = []
cutoff_concentrations = []
scalings_pid_to_cell = []
for ii, gas in enumerate(gases):
    with open(Path("models", f"estimator-{gas.lower()}.pkl"), "rb") as file:
        calibration = pickle.load(file)

    concentration_estimator = calibration["estimator"]

    if calibration_type == "flowrate":
        concentration = concentration_estimator(flowrates[:, ii + 1])
    elif calibration_type == "pid":
        scaling_pid_to_cell = scaling_from_pid_to_cell(flowrates, split_ratio, ii + 1)
        concentration = scaling_pid_to_cell * concentration_estimator(voltages[:, ii])

        cutoff_concentrations.append(calibration["cutoff_concentration"])
        scalings_pid_to_cell.append(scaling_pid_to_cell)

    concentrations.append(concentration)

concentrations = np.stack(concentrations, axis=1)
if calibration_type == "pid":
    cutoff_concentrations = np.array(cutoff_concentrations)
    scalings_pid_to_cell = np.stack(scalings_pid_to_cell, axis=1)

### Define selection mask

In [6]:
selection_mask = are_settled
if calibration_type == "pid":
    selection_mask = selection_mask & np.all(
        concentrations < cutoff_concentrations * scalings_pid_to_cell,
        axis=1,
    )

### Load regression model

In [7]:
with open(Path("models", f"ridge-{gas_pair}.pkl"), "rb") as file:
    model = pickle.load(file)

concentration_predictions = model.predict(absorbances)

### Filter model predictions

In [8]:
with open(Path("models", f"kalman-{gas_pair}.pkl"), "rb") as file:
    kalman_filter = pickle.load(file)

In [9]:
time_per_spectrum = np.mean(np.diff(times))

factors = np.linspace(0, 4, 5)
number_factors = len(factors)

kalman_filter.R = np.cov(concentration_predictions[are_analyte_free, :].T)

concentrations_filtered = np.zeros((number_factors, number_spectra, number_gases))

for ii, factor in enumerate(factors):
    if factor != 0:
        filter_time_constant = factor * rise_time / np.log(9.0)
        filter_gain = 1 - np.exp(-time_per_spectrum / filter_time_constant)

        kalman_filter.x = np.zeros(number_gases)
        kalman_filter.Q = kalman_filter.R * filter_gain**2 / (1 - filter_gain)
        kalman_filter.P = kalman_filter.R * filter_gain

        for jj in range(number_spectra):
            kalman_filter.predict()
            kalman_filter.update(concentration_predictions[jj, :])

            concentrations_filtered[ii, jj, :] = kalman_filter.x

    else:
        concentrations_filtered[ii, :, :] = concentration_predictions

### Characterize dynamic response

In [10]:
setpoints = np.round(setpoints, 1)
maximum_setpoints = np.max(setpoints, axis=0)

channels = np.arange(number_gases)

response_times_cstr = np.zeros(number_gases)

response_times = np.zeros((number_gases, number_factors))
response_times_se = np.zeros((number_gases, number_factors))
limits_of_detection = np.zeros((number_gases, number_factors))
limits_of_detection_se = np.zeros((number_gases, number_factors))

x = np.concatenate(
    [concentrations[selection_mask, :], np.ones((selection_mask.sum(), 1))],
    axis=1,
)

traces = {}

for ii in channels:
    other_channels = channels[channels != ii]

    is_maximal = (setpoints[:, ii + 1] == maximum_setpoints[ii + 1]) & np.all(
        setpoints[:, other_channels + 1] == 0, axis=1
    )

    cycle_time = np.ptp(times[is_maximal])
    switch_time = times[np.min(np.flatnonzero(np.diff(is_maximal)))]

    pre_switch_mask = (times >= switch_time - cycle_time + settling_time) & (
        times <= switch_time
    )
    post_switch_mask = (times >= switch_time + settling_time) & (
        times <= switch_time + cycle_time
    )
    cycle_mask = (times >= switch_time - cycle_time + settling_time) & (
        times <= switch_time + cycle_time
    )

    # Data for current cycle
    time_cycle = times[cycle_mask]

    # gas cell modelled as a continuous stirred-tank reactor
    flowrate = np.max(np.sum(flowrates[cycle_mask], axis=1)) / 60  # (ml / s)
    tau = gas_cell_volume / flowrate

    response_times_cstr[ii] = tau * np.log(9.0)

    for jj, factor in enumerate(factors):
        concentration_filtered = concentrations_filtered[jj, :, ii]

        ### 10-90% response time ###
        c_0 = np.mean(concentration_filtered[pre_switch_mask])
        c_100 = np.mean(concentration_filtered[post_switch_mask])

        c_10 = c_0 + 0.1 * (c_100 - c_0)
        c_90 = c_0 + 0.9 * (c_100 - c_0)

        # crossing times and local slopes
        concentration_cycle = concentration_filtered[cycle_mask]
        t_10, slope_10 = crossing_time_slope(time_cycle, concentration_cycle, c_10)
        t_90, slope_90 = crossing_time_slope(time_cycle, concentration_cycle, c_90)

        # estimate concentration noise from steady-state regions
        sigma_c = np.sqrt(
            (
                np.var(concentration_filtered[pre_switch_mask], ddof=1)
                + np.var(concentration_filtered[post_switch_mask], ddof=1)
            )
            / 2
        )

        # convert concentration noise -> crossing-time uncertainty
        t_10_se = sigma_c / abs(slope_10)
        t_90_se = sigma_c / abs(slope_90)

        # 10–90 % response time and its uncertainty
        response_time = t_90 - t_10
        response_time_se = np.sqrt(t_10_se**2 + t_90_se**2)

        ### limit of detection ###
        fits = sm.OLS(concentration_filtered[selection_mask], x).fit()

        s_ii = fits.params[ii]
        s_ii_se = fits.bse[ii]

        sigma_i = np.std(concentration_filtered[are_analyte_free], ddof=1)
        sigma_i_se = sigma_i / np.sqrt(2 * (are_analyte_free.sum() - 1))

        response_times[ii, jj] = response_time
        response_times_se[ii, jj] = response_time_se
        limits_of_detection[ii, jj] = 3 * sigma_i / s_ii
        limits_of_detection_se[ii, jj] = limits_of_detection[ii, jj] * np.sqrt(
            sigma_i_se**2 / sigma_i**2 + s_ii_se**2 / s_ii**2
        )

        if factor == 1:
            time_offset = np.min(times[cycle_mask])

            trace_time = times[cycle_mask] - time_offset
            trace_switch_time = switch_time - time_offset

            cstr_time = np.linspace(0, np.max(trace_time), 1_001)
            cstr_concentration = np.full_like(cstr_time, c_0)
            is_post_switch = cstr_time >= trace_switch_time
            cstr_concentration[is_post_switch] = c_100 - (c_100 - c_0) * np.exp(
                -(cstr_time[is_post_switch] - trace_switch_time) / tau
            )

            traces[ii] = {
                "time": trace_time,
                "concentration": concentration_filtered[cycle_mask],
                "cstr_time": cstr_time,
                "cstr_concentration": cstr_concentration,
                "response_time_range": np.array(
                    [t_10 - time_offset, t_90 - time_offset]
                ),
            }

            print(
                f"Response time ({gas_labels[gases[ii]]}): {format_number(response_time, response_time_se)} +/- {format_number(response_time_se, response_time_se)} s"
            )

Response time (iPrOH): 85.3 +/- 4.2 s
Response time (EtOH): 92.1 +/- 5.7 s


### Export results

In [11]:
arrays = {
    "factors": factors,
    "response_times_cstr": response_times_cstr,
    "response_times": response_times,
    "response_times_se": response_times_se,
    "limits_of_detection": limits_of_detection,
    "limits_of_detection_se": limits_of_detection_se,
}
for ii in channels:
    arrays.update({f"{key}_{ii}": value for key, value in traces[ii].items()})

metadata = {
    "gases": list(gases),
    "spectral_resolution": float(spectral_resolution),
    "time_per_spectrum": float(time_per_spectrum),
}

filepath = Path("results", f"dynamic-characterization-{configuration}")
filepath.parent.mkdir(exist_ok=True)

np.savez(filepath.with_suffix(".npz"), **arrays)
filepath.with_suffix(".yaml").write_text(yaml.safe_dump(metadata, sort_keys=False))

print(f"Saved: {filepath.with_suffix('.npz')}")
print(f"Saved: {filepath.with_suffix('.yaml')}")

Saved: results\dynamic-characterization-t1.npz
Saved: results\dynamic-characterization-t1.yaml
